# AGENTS026 — Chaos Scheduler
**UC-6: Autonomous fault injection scheduling with GPU-driven scenario planning**

Automatically schedules, fires, monitors, and clears faults across MiniCluster services.
Qwen3-30B selects the next scenario based on current cluster state.

- Cell 1: Imports & config
- Cell 2: Scenario library
- Cell 3: GPU scenario selector
- Cell 4: Chaos executor
- Cell 5: Run one scenario (manual trigger)
- Cell 6: Full autonomous loop (N rounds)
- Cell 7: Results summary

In [1]:
# ── Cell 1: Imports & config ──────────────────────────────────────────────
import json, time, requests, random
from datetime import datetime, timezone
from pathlib import Path
from openai import OpenAI

SERVICES = {"payments": 7001, "auth": 7002, "checkout": 7003, "fraud": 7004}
METRICS_CSV   = Path("/workspace/shared/minicluster/live_metrics.csv")
CHAOS_LOG     = Path("/workspace/shared/chaos_log.jsonl")
AUDIT_FILE    = Path("/workspace/shared/audit_log.jsonl")
HITL_FILE     = Path("/workspace/shared/hitl_queue.jsonl")

llm = OpenAI(base_url="http://localhost:8000/v1", api_key="abc-123")

def ts():
    return datetime.now(timezone.utc).isoformat()

def write_log(path, event):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(event, default=str) + "\n")

def fault_post(port, path, payload):
    try:
        r = requests.post(f"http://127.0.0.1:{port}{path}", json=payload, timeout=3)
        return r.json()
    except Exception as e:
        return {"error": str(e)}

def clear_all():
    for svc, port in SERVICES.items():
        fault_post(port, "/fault/clear", {})
    print("[chaos] All faults cleared")

def get_latest_metrics():
    import pandas as pd
    if not METRICS_CSV.exists():
        return {}
    df = pd.read_csv(METRICS_CSV, parse_dates=["timestamp"])
    return df.sort_values("timestamp").groupby("service").last().to_dict("index")

print("✅ Cell 1 ready — imports & config loaded")

✅ Cell 1 ready — imports & config loaded


In [2]:
# ── Cell 2: Scenario library ──────────────────────────────────────────────
# Each scenario: name, description, faults list, duration_secs, blast_radius

SCENARIOS = [
    {
        "id": "SC-01",
        "name": "Payment Latency Spike",
        "description": "Simulate upstream payment processor slowdown — latency hits checkout cascade",
        "faults": [
            {"service": "payments", "endpoint": "/fault/latency", "payload": {"ms": 800}}
        ],
        "duration_secs": 90,
        "blast_radius": "medium",
        "expected_impact": "checkout latency rises via payment_auth dependency"
    },
    {
        "id": "SC-02",
        "name": "Auth Cascade Failure",
        "description": "Auth service errors propagate to payments and checkout simultaneously",
        "faults": [
            {"service": "auth", "endpoint": "/fault/errors", "payload": {"pct": 0.40}},
            {"service": "auth", "endpoint": "/fault/latency", "payload": {"ms": 600}}
        ],
        "duration_secs": 90,
        "blast_radius": "high",
        "expected_impact": "payments token_verify and checkout session_check both fail"
    },
    {
        "id": "SC-03",
        "name": "Fraud CPU Saturation",
        "description": "Fraud model CPU spike — risk_signal to payments degrades",
        "faults": [
            {"service": "fraud", "endpoint": "/fault/cpu_spin", "payload": {"seconds": 80}}
        ],
        "duration_secs": 90,
        "blast_radius": "low",
        "expected_impact": "fraud CPU > 70%, latency rises, payments risk_signal degrades"
    },
    {
        "id": "SC-04",
        "name": "Memory Leak + Errors (Checkout)",
        "description": "Checkout OOM scenario: slow memory leak combined with rising error rate",
        "faults": [
            {"service": "checkout", "endpoint": "/fault/mem_leak", "payload": {"mb_per_min": 80}},
            {"service": "checkout", "endpoint": "/fault/errors",  "payload": {"pct": 0.25}}
        ],
        "duration_secs": 90,
        "blast_radius": "medium",
        "expected_impact": "checkout memory climbs, error rate rises, triggers AE anomaly"
    },
    {
        "id": "SC-05",
        "name": "Alert Storm — All Services",
        "description": "Simultaneous multi-service degradation — tests UC-4 alert correlation",
        "faults": [
            {"service": "payments",  "endpoint": "/fault/latency", "payload": {"ms": 700}},
            {"service": "auth",      "endpoint": "/fault/errors",  "payload": {"pct": 0.30}},
            {"service": "checkout",  "endpoint": "/fault/latency", "payload": {"ms": 500}},
            {"service": "fraud",     "endpoint": "/fault/errors",  "payload": {"pct": 0.20}}
        ],
        "duration_secs": 120,
        "blast_radius": "critical",
        "expected_impact": "10+ simultaneous alerts — UC-4 NetworkX clustering should group them"
    },
    {
        "id": "SC-06",
        "name": "Silent Fraud Drop",
        "description": "Fraud goes silent — no errors, just stops generating traffic (UC-5 trigger)",
        "faults": [
            {"service": "fraud", "endpoint": "/fault/latency", "payload": {"ms": 3000}}
        ],
        "duration_secs": 90,
        "blast_radius": "low",
        "expected_impact": "fraud appears silent — timeout causes no-ops, UC-5 gap detector fires"
    }
]

print(f"✅ Cell 2 ready — {len(SCENARIOS)} chaos scenarios loaded")
for s in SCENARIOS:
    print(f"  {s['id']} [{s['blast_radius'].upper():8s}] {s['name']}")

✅ Cell 2 ready — 6 chaos scenarios loaded
  SC-01 [MEDIUM  ] Payment Latency Spike
  SC-02 [HIGH    ] Auth Cascade Failure
  SC-03 [LOW     ] Fraud CPU Saturation
  SC-04 [MEDIUM  ] Memory Leak + Errors (Checkout)
  SC-05 [CRITICAL] Alert Storm — All Services
  SC-06 [LOW     ] Silent Fraud Drop


In [3]:
# ── Cell 3: GPU scenario selector ─────────────────────────────────────────
# Qwen3-30B picks the best next scenario based on cluster state

def gpu_select_scenario(metrics, already_run=None):
    already_run = already_run or []
    available = [s for s in SCENARIOS if s["id"] not in already_run]
    if not available:
        return None, "All scenarios have been run."

    scenario_list = "\n".join(
        f"- {s['id']}: {s['name']} [blast={s['blast_radius']}] — {s['description']}"
        for s in available
    )

    prompt = f"""You are an AIOps chaos engineering agent for a banking microservices cluster.
Current live metrics:
{json.dumps(metrics, default=str, indent=2)}

Available chaos scenarios (not yet run):
{scenario_list}

Select the SINGLE best scenario to run next to stress-test the system effectively.
Prefer scenarios that will expose weaknesses visible in the current metrics.
If all services are healthy, prefer medium or high blast-radius scenarios.
If one service is already degraded, avoid making it worse — target a different service.

Respond in JSON only:
{{"selected_id": "SC-XX", "reasoning": "one sentence"}}"""

    try:
        resp = llm.chat.completions.create(
            model="Qwen3-30B-A3B",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3, max_tokens=150,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        raw = resp.choices[0].message.content.strip()
        # Strip markdown fences if present
        raw = raw.replace("```json","").replace("```","").strip()
        result = json.loads(raw)
        sid    = result["selected_id"]
        reason = result["reasoning"]
        scenario = next((s for s in available if s["id"] == sid), available[0])
        return scenario, reason
    except Exception as e:
        # Fallback: pick randomly from available
        scenario = random.choice(available)
        return scenario, f"GPU fallback (random): {e}"

# Quick test
metrics = get_latest_metrics()
scenario, reason = gpu_select_scenario(metrics)
print(f"🧠 GPU selected: {scenario['id']} — {scenario['name']}")
print(f"   Reasoning: {reason}")

🧠 GPU selected: SC-02 — Auth Cascade Failure
   Reasoning: Auth Cascade Failure has a high blast-radius and could expose potential propagation issues in the system, especially since auth is a critical service with no current errors.


In [4]:
# ── Cell 4: Chaos executor ────────────────────────────────────────────────

def run_scenario(scenario, reason="manual", hitl_gate=False):
    """
    Execute a chaos scenario:
    1. Log intent
    2. Optionally gate on HITL
    3. Fire all faults
    4. Wait duration
    5. Snapshot post-fault metrics
    6. Clear faults
    7. Write results
    """
    run_id = f"chaos-{scenario['id']}-{int(time.time())}"
    print(f"\n{'='*60}")
    print(f"🔥 CHAOS RUN: {run_id}")
    print(f"   Scenario : {scenario['name']}")
    print(f"   Blast    : {scenario['blast_radius'].upper()}")
    print(f"   Reason   : {reason}")
    print(f"   Duration : {scenario['duration_secs']}s")
    print(f"{'='*60}")

    intent = {
        "run_id": run_id,
        "scenario_id": scenario["id"],
        "scenario_name": scenario["name"],
        "blast_radius": scenario["blast_radius"],
        "gpu_reasoning": reason,
        "faults": scenario["faults"],
        "status": "PENDING_HITL" if hitl_gate else "RUNNING",
        "timestamp": ts()
    }
    write_log(CHAOS_LOG, intent)

    # HITL gate for critical blast radius
    if hitl_gate and scenario["blast_radius"] == "critical":
        hitl_evt = {
            "hitl_id": run_id,
            "timestamp": ts(),
            "source": "chaos_scheduler",
            "service": "ALL",
            "scenario": scenario["name"],
            "blast_radius": scenario["blast_radius"],
            "faults": scenario["faults"],
            "status": "PENDING"
        }
        write_log(HITL_FILE, hitl_evt)
        write_log(AUDIT_FILE, {**hitl_evt, "event_type": "CHAOS_HITL_CREATED"})
        print(f"⏳ HITL gate raised for CRITICAL scenario — check HITL Queue tab")
        return {"status": "HITL_PENDING", "run_id": run_id}

    # Snapshot pre-fault metrics
    pre_metrics = get_latest_metrics()

    # Fire faults
    fault_results = []
    for fault in scenario["faults"]:
        port = SERVICES[fault["service"]]
        result = fault_post(port, fault["endpoint"], fault["payload"])
        fault_results.append({"service": fault["service"], "endpoint": fault["endpoint"], "result": result})
        print(f"   💥 {fault['service']:10s} {fault['endpoint']:20s} → {result}")

    write_log(AUDIT_FILE, {
        "event_type": "CHAOS_FAULTS_FIRED",
        "run_id": run_id,
        "faults": fault_results,
        "timestamp": ts()
    })

    # Wait and sample
    print(f"\n⏱️  Waiting {scenario['duration_secs']}s for metrics to stabilise...")
    for i in range(0, scenario["duration_secs"], 10):
        time.sleep(10)
        pct = int((i + 10) / scenario["duration_secs"] * 20)
        bar = "█" * pct + "░" * (20 - pct)
        elapsed = i + 10
        print(f"   [{bar}] {elapsed}/{scenario['duration_secs']}s", end="\r")

    print("\n")

    # Post-fault snapshot
    post_metrics = get_latest_metrics()

    # Compute deltas
    deltas = {}
    for svc in SERVICES:
        pre  = pre_metrics.get(svc, {})
        post = post_metrics.get(svc, {})
        deltas[svc] = {
            "cpu_delta":     round(float(post.get("cpu_utilization",0)) - float(pre.get("cpu_utilization",0)), 2),
            "latency_delta": round(float(post.get("latency_p95_ms",0))  - float(pre.get("latency_p95_ms",0)),  2),
            "error_delta":   round(float(post.get("error_rate",0))       - float(pre.get("error_rate",0)),       4),
        }

    # GPU post-mortem
    pm_prompt = f"""Banking SRE chaos post-mortem. Scenario: {scenario['name']}.
Expected: {scenario['expected_impact']}
Metric deltas (post minus pre): {json.dumps(deltas)}
Write ONE sentence: did the scenario behave as expected, and what is the key finding?"""

    try:
        pm_resp = llm.chat.completions.create(
            model="Qwen3-30B-A3B",
            messages=[{"role": "user", "content": pm_prompt}],
            temperature=0.2, max_tokens=80,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        post_mortem = pm_resp.choices[0].message.content.strip()
    except Exception as e:
        post_mortem = f"GPU unavailable: {e}"

    # Clear faults
    clear_all()

    # Write result
    result_record = {
        "run_id":       run_id,
        "scenario_id":  scenario["id"],
        "scenario_name": scenario["name"],
        "blast_radius": scenario["blast_radius"],
        "gpu_reasoning": reason,
        "pre_metrics":  pre_metrics,
        "post_metrics": post_metrics,
        "deltas":       deltas,
        "post_mortem":  post_mortem,
        "fault_results": fault_results,
        "status":       "COMPLETE",
        "timestamp":    ts()
    }
    write_log(CHAOS_LOG, result_record)
    write_log(AUDIT_FILE, {**result_record, "event_type": "CHAOS_RUN_COMPLETE"})

    print(f"✅ Run complete: {run_id}")
    print(f"🧠 Post-mortem: {post_mortem}")
    print(f"📊 Deltas: {json.dumps(deltas, indent=2)}")
    return result_record

print("✅ Cell 4 ready — chaos executor loaded")

✅ Cell 4 ready — chaos executor loaded


In [5]:
# ── Cell 5: Run ONE scenario (manual trigger) ─────────────────────────────
# Choose a specific scenario by ID, or let GPU pick

FORCE_SCENARIO = None   # Set to e.g. "SC-02" to force, or None for GPU pick
HITL_FOR_CRITICAL = True  # Gate HITL approval for blast_radius=critical

metrics = get_latest_metrics()

if FORCE_SCENARIO:
    scenario = next(s for s in SCENARIOS if s["id"] == FORCE_SCENARIO)
    reason   = "manual override"
else:
    print("🧠 Asking GPU to select scenario...")
    scenario, reason = gpu_select_scenario(metrics)

print(f"\n▶️  About to run: {scenario['id']} — {scenario['name']}")
print(f"   Blast radius : {scenario['blast_radius']}")
print(f"   Duration     : {scenario['duration_secs']}s")
print(f"   Faults       : {len(scenario['faults'])}")
print(f"   Reasoning    : {reason}")
print()

result = run_scenario(scenario, reason=reason, hitl_gate=HITL_FOR_CRITICAL)

🧠 Asking GPU to select scenario...

▶️  About to run: SC-02 — Auth Cascade Failure
   Blast radius : high
   Duration     : 90s
   Faults       : 2
   Reasoning    : Auth Cascade Failure has a high blast radius and could expose potential propagation issues in the system, especially since auth is a critical service with no current errors.


🔥 CHAOS RUN: chaos-SC-02-1781379574
   Scenario : Auth Cascade Failure
   Blast    : HIGH
   Reason   : Auth Cascade Failure has a high blast radius and could expose potential propagation issues in the system, especially since auth is a critical service with no current errors.
   Duration : 90s
   💥 auth       /fault/errors        → {'fault': 'errors', 'pct': 0.2}
   💥 auth       /fault/latency       → {'fault': 'latency', 'ms': 500}

⏱️  Waiting 90s for metrics to stabilise...
   [████████████████████] 90/90s

[chaos] All faults cleared
✅ Run complete: chaos-SC-02-1781379574
🧠 Post-mortem: The scenario did not behave as expected, as the auth service

In [6]:
# ── Cell 6: Full autonomous loop (N rounds) ───────────────────────────────
# GPU picks scenario each round, runs it, records results
# Skips CRITICAL scenarios unless ALLOW_CRITICAL=True

N_ROUNDS      = 3       # How many scenarios to run end-to-end
ALLOW_CRITICAL = False  # Set True to allow SC-05 (alert storm)
COOLDOWN_SECS  = 30     # Gap between scenarios for metrics to normalise

already_run = []
all_results = []

print(f"🤖 AUTONOMOUS CHAOS LOOP — {N_ROUNDS} rounds")
print(f"   Allow critical: {ALLOW_CRITICAL}")
print(f"   Cooldown: {COOLDOWN_SECS}s between rounds")
print()

for round_num in range(1, N_ROUNDS + 1):
    print(f"\n{'━'*60}")
    print(f"ROUND {round_num}/{N_ROUNDS}")
    print(f"{'━'*60}")

    metrics  = get_latest_metrics()
    scenario, reason = gpu_select_scenario(metrics, already_run=already_run)

    if scenario is None:
        print("All scenarios exhausted — stopping loop.")
        break

    if scenario["blast_radius"] == "critical" and not ALLOW_CRITICAL:
        print(f"⚠️  Skipping {scenario['id']} (critical, ALLOW_CRITICAL=False)")
        already_run.append(scenario["id"])
        continue

    result = run_scenario(scenario, reason=reason, hitl_gate=True)
    all_results.append(result)
    already_run.append(scenario["id"])

    if round_num < N_ROUNDS:
        print(f"\n⏸️  Cooldown {COOLDOWN_SECS}s before next round...")
        time.sleep(COOLDOWN_SECS)

print(f"\n🏁 Autonomous loop complete — {len(all_results)} scenarios executed")

🤖 AUTONOMOUS CHAOS LOOP — 3 rounds
   Allow critical: False
   Cooldown: 30s between rounds


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ROUND 1/3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔥 CHAOS RUN: chaos-SC-01-1781379695
   Scenario : Payment Latency Spike
   Blast    : MEDIUM
   Reason   : SC-01 introduces a latency spike in payments, which could expose potential cascading failures in checkout and other dependent services, given the current high RPS and low latency in payments.
   Duration : 90s
   💥 payments   /fault/latency       → {'fault': 'latency', 'ms': 500}

⏱️  Waiting 90s for metrics to stabilise...
   [████████████████████] 90/90s

[chaos] All faults cleared
✅ Run complete: chaos-SC-01-1781379695
🧠 Post-mortem: The scenario did not behave as expected, as the payment_auth dependency was not the source of the latency spike, with the key finding being that the payments service experienced a significant latency increase despite no corre

KeyboardInterrupt: 

In [7]:
# ── Cell 7: Results summary ───────────────────────────────────────────────
import pandas as pd

if not CHAOS_LOG.exists():
    print("No chaos log yet — run Cell 5 or 6 first.")
else:
    records = [json.loads(l) for l in CHAOS_LOG.read_text().strip().split("\n") if l.strip()]
    complete = [r for r in records if r.get("status") == "COMPLETE"]

    print(f"📊 CHAOS RESULTS SUMMARY")
    print(f"   Total runs logged : {len(records)}")
    print(f"   Completed runs    : {len(complete)}")
    print()

    for r in complete:
        print(f"  {'─'*55}")
        print(f"  {r['run_id']}")
        print(f"  Scenario : {r['scenario_name']}")
        print(f"  Blast    : {r['blast_radius']}")
        print(f"  Reason   : {r.get('gpu_reasoning','')}")
        print(f"  Verdict  : {r.get('post_mortem','')}")
        if r.get("deltas"):
            worst = max(r["deltas"].items(),
                        key=lambda kv: abs(kv[1].get("latency_delta",0)))
            print(f"  Worst Δ  : {worst[0]} latency +{worst[1]['latency_delta']:.0f}ms")
        print()

    # Build summary dataframe
    if complete:
        rows = []
        for r in complete:
            row = {
                "run_id":    r["run_id"],
                "scenario":  r["scenario_name"],
                "blast":     r["blast_radius"],
                "verdict":   r.get("post_mortem","")[:80],
                "timestamp": r.get("timestamp","")
            }
            rows.append(row)
        df = pd.DataFrame(rows)
        print(df.to_string(index=False))

📊 CHAOS RESULTS SUMMARY
   Total runs logged : 7
   Completed runs    : 3

  ───────────────────────────────────────────────────────
  chaos-SC-02-1781379574
  Scenario : Auth Cascade Failure
  Blast    : high
  Reason   : Auth Cascade Failure has a high blast radius and could expose potential propagation issues in the system, especially since auth is a critical service with no current errors.
  Verdict  : The scenario did not behave as expected, as the auth service exhibited a significant latency increase and error rate, while payments and checkout did not show the anticipated failures, indicating a potential misalignment in the cascading failure impact.
  Worst Δ  : auth latency +499ms

  ───────────────────────────────────────────────────────
  chaos-SC-01-1781379695
  Scenario : Payment Latency Spike
  Blast    : medium
  Reason   : SC-01 introduces a latency spike in payments, which could expose potential cascading failures in checkout and other dependent services, given the curre